# TDRNN on RML2018.01a Dataset

Key differences from RML2016.10a:
- **File format**: HDF5 (`.hdf5`) instead of pickle
- **Signal length**: 1024 samples per signal (vs 128 in 2016)
- **Classes**: 24 modulation types (vs 11)
- **Model**: Conv kernel sizes and GRU input size adapted for length-1024 signals

In [ ]:
import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Load RML2018.01a

The 2018 dataset is distributed as an HDF5 file. It contains three datasets:
- `X`: signals of shape `(N, 2, 1024)` — I/Q channels, 1024 samples each
- `Y`: one-hot encoded labels of shape `(N, 24)`
- `Z`: SNR values of shape `(N, 1)`

In [ ]:
import json, h5py, numpy as np

path = "/kaggle/input/datasets/pinxau1000/radioml2018/GOLD_XYZ_OSC.0001_1024.hdf5"

with open("/kaggle/input/datasets/pinxau1000/radioml2018/classes-fixed.json") as f:
    class_names = json.load(f)

# Load only 200,000 samples — fast, fits easily in RAM
N = 200_000

with h5py.File(path, 'r') as f:
    print("Total samples in file:", f['X'].shape[0])
    X        = f['X'][:N]
    Y_onehot = f['Y'][:N]
    Z        = f['Z'][:N].flatten()

y_encoded   = np.argmax(Y_onehot, axis=1)
snr_list    = Z.astype(int)
num_classes = Y_onehot.shape[1]

X = X.reshape(-1, 1, 2, 1024)

print("Loaded :", X.shape)
print("Classes:", num_classes)
print("SNR    :", snr_list.min(), "to", snr_list.max(), "dB")

## Train / Val / Test split

In [ ]:
X_train, X_temp, y_train, y_temp, snr_train, snr_temp = train_test_split(
    X, y_encoded, snr_list, test_size=0.4, stratify=y_encoded, random_state=42
)

X_val, X_test, y_val, y_test, snr_val, snr_test = train_test_split(
    X_temp, y_temp, snr_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print("Train:", X_train.shape)
print("Val  :", X_val.shape)
print("Test :", X_test.shape)

## DataLoaders

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val,   dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val,   dtype=torch.long)
y_test  = torch.tensor(y_test,  dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=1024, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=1024)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=1024)

## TDRNN Model  (adapted for signal length 1024)

The only structural change from the 2016 version:
- `conv3` now uses `padding=(0,1)` (instead of `(0,0)`) so the spatial width
  stays wide enough after three convolutions on a 1024-sample signal.
- Everything else is identical.

In [ ]:
class TDRNN(nn.Module):
    def __init__(self, num_classes=24, gru_hidden=64):
        super(TDRNN, self).__init__()

        self.bn = nn.BatchNorm2d(1)

        self.conv1 = nn.Conv2d(1,  16, kernel_size=(1, 3), padding=(0, 1))
        self.conv2 = nn.Conv2d(16, 32, kernel_size=(2, 3), padding=(0, 1))
        # For 1024-sample signals keep padding so width doesn't collapse
        self.conv3 = nn.Conv2d(32, 64, kernel_size=(1, 3), padding=(0, 1))

        self.gap   = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1   = nn.Linear(16, 16)
        self.bn_fc = nn.BatchNorm1d(16)
        self.fc2   = nn.Linear(16, 16)

        self.gru = nn.GRU(input_size=64, hidden_size=gru_hidden, batch_first=True)
        self.fc  = nn.Linear(gru_hidden, num_classes)

    def threshold_block(self, x):
        abs_x = torch.abs(x)

        beta = self.gap(abs_x)
        beta = beta.squeeze(-1).squeeze(-1)

        beta = F.relu(self.fc1(beta))
        beta = self.bn_fc(beta)
        beta = torch.sigmoid(self.fc2(beta))

        beta = beta.unsqueeze(2).unsqueeze(3)
        tau  = 2 * beta

        return torch.sign(x) * F.relu(abs_x - tau)

    def forward(self, x):
        x = self.bn(x)

        x = F.relu(self.conv1(x))   # (B, 16, 2, 1024)
        x = self.threshold_block(x)

        x = F.relu(self.conv2(x))   # (B, 32, 1, 1024)
        x = F.relu(self.conv3(x))   # (B, 64, 1, 1024)

        x = x.squeeze(2)            # (B, 64, 1024)
        x = x.permute(0, 2, 1)      # (B, 1024, 64)

        _, h = self.gru(x)          # h: (1, B, gru_hidden)

        return self.fc(h[-1])

# Quick sanity check
_m = TDRNN(num_classes=24, gru_hidden=64).to(device)
_x = torch.randn(2, 1, 2, 1024).to(device)
print("Output shape:", _m(_x).shape)   # should be (2, 24)
del _m, _x

## Training loop

In [ ]:
def train_model(model, optimizer, epochs=30, save_path="tdrnn_main.pth"):
    criterion = nn.CrossEntropyLoss()

    best_loss = float('inf')
    patience  = 10
    counter   = 0

    for epoch in range(epochs):
        model.train()
        train_loss, correct, total = 0, 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, pred     = torch.max(outputs, 1)
            total      += labels.size(0)
            correct    += (pred == labels).sum().item()

        train_acc = 100 * correct / total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)

                val_loss    += criterion(outputs, labels).item()
                _, pred      = torch.max(outputs, 1)
                val_total   += labels.size(0)
                val_correct += (pred == labels).sum().item()

        val_acc = 100 * val_correct / val_total

        print(f"Epoch {epoch+1}: Train={train_acc:.2f}%, Val={val_acc:.2f}%")

        if val_loss < best_loss:
            best_loss = val_loss
            counter   = 0
            torch.save(model.state_dict(), save_path)
        else:
            counter += 1
            if counter >= patience:
                print("Early stopping")
                break

## SNR-wise accuracy helper

In [ ]:
def snr_accuracy(model):
    model.eval()
    snr_correct, snr_total = {}, {}

    with torch.no_grad():
        for i, (inputs, labels) in enumerate(test_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, pred = torch.max(outputs, 1)

            batch_snrs = snr_test[i * 1024 : i * 1024 + inputs.size(0)]

            for j in range(len(labels)):
                snr = int(batch_snrs[j])
                snr_total[snr]   = snr_total.get(snr, 0) + 1
                if pred[j] == labels[j]:
                    snr_correct[snr] = snr_correct.get(snr, 0) + 1

    snrs = sorted(snr_total.keys())
    acc  = [100 * snr_correct.get(s, 0) / snr_total[s] for s in snrs]

    return snrs, acc

## Train main TDRNN (with denoiser)

In [ ]:
model     = TDRNN(num_classes, gru_hidden=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

train_model(model, optimizer, epochs=30, save_path="tdrnn_main.pth")

model.load_state_dict(torch.load("tdrnn_main.pth"))

## Plot SNR accuracy (single model)

In [ ]:
snrs, acc = snr_accuracy(model)

for s, a in zip(snrs, acc):
    print(f"SNR {s:+3d} dB : {a:.2f}%")

plt.figure(figsize=(12, 6))
plt.plot(snrs, acc, marker='o')
plt.xlabel("SNR (dB)")
plt.ylabel("Accuracy (%)")
plt.title("TDRNN Accuracy vs SNR  [RML2018.01a]")
plt.grid(True)
plt.savefig("snr_plot_2018.png")
plt.show()

## TDRNN without denoiser (ablation)

In [ ]:
class TDRNN_NoDenoise(TDRNN):
    def forward(self, x):
        x = self.bn(x)

        x = F.relu(self.conv1(x))   # skip threshold_block

        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))

        x = x.squeeze(2)
        x = x.permute(0, 2, 1)

        _, h = self.gru(x)

        return self.fc(h[-1])

In [ ]:
model_no     = TDRNN_NoDenoise(num_classes, gru_hidden=64).to(device)
optimizer_no = torch.optim.Adam(model_no.parameters(), lr=0.001)

train_model(model_no, optimizer_no, epochs=30, save_path="tdrnn_no_denoise.pth")

model_no.load_state_dict(torch.load("tdrnn_no_denoise.pth"))

In [ ]:
snr1, acc1 = snr_accuracy(model)      # WITH denoiser
snr2, acc2 = snr_accuracy(model_no)   # WITHOUT denoiser

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(snr1, acc1, marker='o',  linewidth=2, label="TDRNN with Denoiser")
plt.plot(snr2, acc2, marker='^',  linewidth=2, label="TDRNN without Denoiser")

plt.xlabel("SNR (dB)")
plt.ylabel("Accuracy (%)")
plt.title("TDRNN with vs without Threshold Denoiser  [RML2018.01a]")
plt.grid(True)
plt.legend()
plt.savefig("fig6_comparison_2018.png")
plt.show()

## GRU hidden-size sweep

In [ ]:
gru_sizes = [16, 32, 64, 128, 256]

In [ ]:
gru_results = {}

for g in gru_sizes:
    print(f"\n🔹 Training GRU hidden={g}")

    model_g    = TDRNN(num_classes, gru_hidden=g).to(device)
    optimizer_g = torch.optim.Adam(model_g.parameters(), lr=0.001)

    train_model(model_g, optimizer_g, epochs=60, save_path=f"tdrnn_gru{g}.pth")

    model_g.load_state_dict(torch.load(f"tdrnn_gru{g}.pth"))

    snrs, acc = snr_accuracy(model_g)
    gru_results[g] = (snrs, acc)

In [ ]:
plt.figure(figsize=(12, 6))

markers = ['o', 's', '>', 'o', 'x']
colors  = ['green', 'red', 'blue', 'black', 'orange']

for i, g in enumerate(gru_sizes):
    snrs, acc = gru_results[g]
    avg_acc   = sum(acc) / len(acc)

    plt.plot(
        snrs, acc,
        marker=markers[i],
        color=colors[i],
        linewidth=2,
        label=f"GRU {g} [{avg_acc:.1f}%]"
    )

plt.xlabel("SNR (dB)")
plt.ylabel("Accuracy (%)")
plt.title("Classification performance — different GRU hidden sizes  [RML2018.01a]")
plt.grid(True)
plt.legend()
plt.savefig("fig4_gru_comparison_2018.png")
plt.show()

## Model complexity vs GRU size

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def measure_time(model):
    model.eval()
    start = time.time()
    total = 0

    with torch.no_grad():
        for inputs, _ in test_loader:
            inputs = inputs.to(device)
            model(inputs)
            total += inputs.size(0)

    return (time.time() - start) / total * 1000  # ms / sample

In [ ]:
params = []
times  = []

for g in gru_sizes:
    model_g = TDRNN(num_classes, gru_hidden=g).to(device)

    p = count_parameters(model_g)
    t = measure_time(model_g)

    print(f"GRU {g}: Params={p:,}, Time={t:.4f} ms")

    params.append(p)
    times.append(t)

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.bar([str(g) for g in gru_sizes], params, color='steelblue', alpha=0.7)
ax1.set_xlabel("GRU Hidden Size")
ax1.set_ylabel("Number of Parameters", color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

ax2 = ax1.twinx()
ax2.plot([str(g) for g in gru_sizes], times, color='orange', marker='o', linewidth=2)
ax2.set_ylabel("Inference Time (ms)", color='orange')
ax2.tick_params(axis='y', labelcolor='orange')

plt.title("Model Complexity vs GRU Hidden Size  [RML2018.01a]")
fig.tight_layout()
plt.savefig("fig5_complexity_2018.png")
plt.show()